In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("Bakery_1.csv")

In [3]:
df

,TransactionNo,Items,DateTime,Daypart,DayType
0,1,Bread,2016-10-30 09:58:11,Morning,Weekend
1,2,Scandinavian,2016-10-30 10:05:34,Morning,Weekend
2,2,Scandinavian,2016-10-30 10:05:34,Morning,Weekend
3,3,Hot chocolate,2016-10-30 10:07:57,Morning,Weekend
4,3,Jam,2016-10-30 10:07:57,Morning,Weekend
...,...,...,...,...,...
20502,9682,Coffee,2017-09-04 14:32:58,Afternoon,Weekend
20503,9682,Tea,2017-09-04 14:32:58,Afternoon,Weekend
20504,9683,Coffee,2017-09-04 14:57:06,Afternoon,Weekend
20505,9683,Pastry,2017-09-04 14:57:06,Afternoon,Weekend


In [4]:
from mlxtend.frequent_patterns import apriori, association_rules 

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20507 entries, 0 to 20506
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   TransactionNo  20507 non-null  int64 
 1   Items          20507 non-null  object
 2   DateTime       20507 non-null  object
 3   Daypart        20507 non-null  object
 4   DayType        20507 non-null  object
dtypes: int64(1), object(4)
memory usage: 801.2+ KB


In [6]:
df.describe()

,TransactionNo
count,20507.000000
mean,4976.202370
std,2796.203001
min,1.000000
25%,2552.000000
50%,5137.000000
75%,7357.000000
max,9684.000000


In [7]:
df['Items'].nunique()

94

In [8]:
basket = (df.groupby(['TransactionNo', 'Items'])['Items']
            .count().unstack().reset_index().fillna(0)
            .set_index('TransactionNo'))


In [9]:
basket = basket.applymap(lambda x: 1 if x > 0 else 0)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8336\3498954818.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  basket = basket.applymap(lambda x: 1 if x > 0 else 0)


In [10]:
frequent_itemsets = apriori(basket, min_support=0.01, use_colnames=True)  

C:\Users\Lenovo\anaconda3\Lib\site-packages\mlxtend\frequent_patterns\fpcommon.py:161: DeprecationWarning: DataFrames with non-bool types result in worse computationalperformance and their support might be discontinued in the future.Please use a DataFrame with bool type
  warnings.warn(


In [11]:
rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

In [15]:
rules = rules.sort_values(by="confidence","lift", ascending=[False,False])

In [16]:
print(rules.head(10))

        antecedents consequents  antecedent support  consequent support  \
19          (Toast)    (Coffee)            0.033597            0.478394   
12      (Medialuna)    (Coffee)            0.061807            0.478394   
14         (Pastry)    (Coffee)            0.086107            0.478394   
10          (Juice)    (Coffee)            0.038563            0.478394   
16       (Sandwich)    (Coffee)            0.071844            0.478394   
3            (Cake)    (Coffee)            0.103856            0.478394   
6         (Cookies)    (Coffee)            0.054411            0.478394   
8   (Hot chocolate)    (Coffee)            0.058320            0.478394   
0          (Pastry)     (Bread)            0.086107            0.327205   
4            (Cake)       (Tea)            0.103856            0.142631   

     support  confidence      lift  representativity  leverage  conviction  \
19  0.023666    0.704403  1.472431               1.0  0.007593    1.764582   
12  0.035182    0.

In [17]:
df['Context'] = df['Daypart'] + "_" + df['DayType']

In [18]:
basket = (df.groupby(['TransactionNo', 'Context'])['Items']
            .apply(list)
            .reset_index())

In [19]:
from mlxtend.preprocessing import TransactionEncoder

te = TransactionEncoder()
te_ary = te.fit(basket['Items']).transform(basket['Items'])
item_matrix = pd.DataFrame(te_ary, columns=te.columns_)

In [20]:
item_matrix['Context'] = basket['Context']

In [21]:
rules_list = []

for context in item_matrix['Context'].unique():
    sub_matrix = item_matrix[item_matrix['Context'] == context].drop(columns=['Context'])
    
    
    frequent_items = apriori(sub_matrix, min_support=0.02, use_colnames=True)
    

    rules = association_rules(frequent_items, metric="lift", min_threshold=1.0)
    rules['Context'] = context
    rules_list.append(rules)

In [34]:
all_rules = pd.concat(rules_list, ignore_index=True)


all_rules = all_rules.sort_values(by=['Context','confidence'], ascending=[True, False])

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_8336\293120780.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  all_rules = pd.concat(rules_list, ignore_index=True)


In [35]:
pd.set_option("display.max_rows", 20)
print(all_rules[['antecedents','consequents','support','confidence','lift','Context']])

         antecedents      consequents   support  confidence      lift  \
66          (Pastry)         (Coffee)  0.025263    0.531646  1.160684   
61            (Cake)         (Coffee)  0.066165    0.527578  1.151803   
58       (Alfajores)         (Coffee)  0.021053    0.518519  1.132025   
68        (Sandwich)         (Coffee)  0.061353    0.511278  1.116218   
64   (Hot chocolate)         (Coffee)  0.027970    0.508197  1.109491   
..               ...              ...       ...         ...       ...   
9           (Coffee)          (Juice)  0.023368    0.047820  1.419960   
5           (Coffee)        (Cookies)  0.022680    0.046414  1.274182   
13          (Coffee)         (Muffin)  0.021993    0.045007  1.023207   
136  (Mineral water)          (Juice)  0.333333    1.000000  3.000000   
137          (Juice)  (Mineral water)  0.333333    1.000000  3.000000   

               Context  
66   Afternoon_Weekday  
61   Afternoon_Weekday  
58   Afternoon_Weekday  
68   Afternoon_Weekday 